# The cost of the statistics

Three statistics were added to the fold: a sweep cost for each of several order sizes, the
order flow contribution of each event, and the volume and value each message traded.  This
notebook asks what they cost.

The method is the one in `why-the-tick-array-book-is-not-faster.ipynb`: two market regimes,
five book variants, best-of timings, and a table rather than a chart wherever a number is
the point.  What is different here is the subject.  That notebook varies the *book*; this
one holds the book fixed and varies what is read off it.

The estimates written in the plan before any of this ran are quoted beside the
measurements, including where they were wrong.

In [1]:
import timeit

import numpy as np
import pandas as pd

from unito26.lob import benchmark, config
from unito26.lob.messages import BUY, SELL, GridDepth, ReportedDepth, SweepSize
from unito26.lob.orderbook import AXIS_B_VARIANTS, AggregateBook, TickArrayBook, _sweep_cost
from unito26.lob.session import MarketSession, SessionStatistics

REPEAT = 5
DEPTH = ReportedDepth(10)
PRICE_UNIT = 100
LEVELS = tuple(GridDepth(n) for n in (1, 2, 3, 5, 10))
SWEEPS = tuple(SweepSize(q) for q in (100, 400, 1600))
WINDOWS = (1, 10, 60)

SPECS = {
    "old only": SessionStatistics(LEVELS, (), ()),
    "+ sweep": SessionStatistics(LEVELS, SWEEPS, ()),
    "+ windows": SessionStatistics(LEVELS, SWEEPS, WINDOWS),
}
SPEC = SPECS["+ windows"]

REGIMES = {
    "shallow": benchmark.session("shallow", config.shallow_mark_params(), 1200.0, seed=11),
    "deep": benchmark.session("deep", config.deep_mark_params(), 1200.0, seed=11),
}
pd.DataFrame({
    name: {"messages": len(s.messages), "occupied levels": s.occupied_levels()}
    for name, s in REGIMES.items()
})

,shallow,deep
messages,34802,34606
occupied levels,39,378


## 1. The floor

Folding with the statistics off: the book is driven, the LOBSTER rows are written, and
nothing else happens.  Every number below is read against this.

In [2]:
floor = pd.DataFrame({
    regime: {
        cls.__name__: benchmark.time_fold(cls, s, DEPTH, SPEC, False, REPEAT)
        for cls in AXIS_B_VARIANTS
    }
    for regime, s in REGIMES.items()
})
floor.style.format("{:.3f} s")

,shallow,deep
AggregateBook,0.300 s,0.877 s
CachedBestBook,0.275 s,0.755 s
HeapBook,0.287 s,0.769 s
BitmapBook,0.292 s,0.765 s
TickArrayBook,0.249 s,0.257 s


## 2. What `apply` is worth

The fold could have recovered the traded volume and value from the fills, by asking the
book to `record`.  It does not: it reads three integers the matching loop accumulates
anyway.  This cell is why.

Two numbers are needed and neither means anything alone.  Recording roughly doubles or
triples `apply` — and `apply` is a few per cent of the fold, so the fold-level cost is the
product, not the ratio.  The plan's first draft guessed "tens of per cent" from the ratio
and was wrong by a factor of three.

In [3]:
rows = {}
for regime, s in REGIMES.items():
    for cls in (AggregateBook, TickArrayBook):
        off = benchmark.time_apply(cls, s, False, REPEAT)
        on = benchmark.time_apply(cls, s, True, REPEAT)
        whole = benchmark.time_fold(cls, s, DEPTH, SPEC, True, REPEAT)
        rows[(regime, cls.__name__)] = {
            "apply, record=False": off,
            "apply, record=True": on,
            "recording costs apply": on / off - 1,
            "apply's share of the fold": off / whole,
            "so, of the fold": (on - off) / whole,
        }
worth = pd.DataFrame(rows).T
worth.style.format({
    "apply, record=False": "{:.3f} s", "apply, record=True": "{:.3f} s",
    "recording costs apply": "{:+.0%}", "apply's share of the fold": "{:.1%}",
    "so, of the fold": "{:+.1%}",
})

## 3. One family at a time

`old only` is the statistics as they were: spread, mid, micro-price, the imbalance profile
and the gap statistics.  Then the sweep costs, then the windows.

The windows are expected to be nearly free *in the fold*, because a window is a reduction
over a recorded series and is made once at assembly.  What they cost here is that single
reduction, not per-message work.  If this column is not close to flat, something is being
computed per message that should not be.

In [4]:
families = {}
for regime, s in REGIMES.items():
    for cls in AXIS_B_VARIANTS:
        base = floor.loc[cls.__name__, regime]
        row = {"statistics off": base}
        for label, spec in SPECS.items():
            row[label] = benchmark.time_fold(cls, s, DEPTH, spec, True, REPEAT)
        families[(regime, cls.__name__)] = row
family = pd.DataFrame(families).T
relative = family.div(family["statistics off"], axis=0)
pd.concat({"seconds": family, "x floor": relative}, axis=1).style.format("{:.2f}")

## 4. The sweep, two ways

`_write_statistics` receives each side as a `SideStatistics`, whose `.levels` *is*
`occupied_levels(direction, reported_depth)` — the walk has already been paid for.  The
alternative is to walk the side again per sweep.

The two shapes below differ only in that.  The batched walk answers every size from one
pass; the naive one restarts per size.  Both are timed against a re-walk and against the
levels already in hand.

In [5]:
def read_sides(book, depth):
    """What `_write_statistics` pays before any sweep: one walk of each side."""
    return book.side_statistics(BUY, depth), book.side_statistics(SELL, depth)


def take(book, depth, sizes, reuse, batched):
    bid, ask = read_sides(book, depth)
    mid = book.mid_price
    for side, levels, direction in (
        (ask, ask.levels if reuse else book.occupied_levels(SELL, depth), BUY),
        (bid, bid.levels if reuse else book.occupied_levels(BUY, depth), SELL),
    ):
        if batched:
            _sweep_cost(levels, mid, sizes, direction)
        else:
            for size in sizes:
                _sweep_cost(levels, mid, (size,), direction)


def best(call):
    return min(timeit.repeat(lambda: [call() for _ in range(200)], number=1, repeat=REPEAT))


def sweep_table(regime, depths, counts):
    s = REGIMES[regime]
    book = TickArrayBook.for_prices(s.prices)
    for message in s.messages:
        book.apply(message, record=False)
    out = {}
    for depth in depths:
        reported = ReportedDepth(depth)
        # The two side walks are paid whether or not a sweep is asked for, so they are the
        # floor here.  Timing the sweep on top of them, rather than the whole read, is what
        # makes the two shapes comparable.
        floor_here = best(lambda: read_sides(book, reported))
        for count in counts:
            sizes = tuple(SweepSize(q) for q in np.linspace(50, 3000, count).astype(int))
            row = {"side walks (floor)": floor_here}
            for reuse in (False, True):
                for batched in (False, True):
                    name = ("reuse" if reuse else "re-walk") + (
                        ", batched" if batched else ", naive")
                    row[name] = best(
                        lambda r=reuse, b=batched: take(book, reported, sizes, r, b)
                    ) - floor_here
            out[(depth, count)] = row
    return pd.DataFrame(out).T

sweeps = sweep_table("deep", (1, 5, 10, 50), (1, 3, 10))
sweeps.index.names = ["reported depth", "sweep sizes"]
sweeps.style.format("{:.4f} s")

## 5. Coverage: what the columns actually hold

A sweep size is an absolute number of shares, and a thin book does not hold it.  Where the
reported levels cannot fill the size the column is NaN, and `…Covered` says whether that
NaN is an answer — the book cannot fill it at any price — or an absence.

A column that is empty most of the time is not the statistic a reader thinks they are
reading, which is why this sits next to the timings rather than in an appendix.

In [6]:
coverage = {}
for regime, s in REGIMES.items():
    for depth in (1, 3, 10):
        session = MarketSession.from_occupied_levels(
            TickArrayBook.for_prices(s.prices), s.messages,
            ReportedDepth(depth), SPEC, PRICE_UNIT, True,
        )
        row = {}
        for size in SWEEPS:
            column = session.stats[f"SweepCostBuy{size}"]
            flag = session.stats[f"SweepCostBuy{size}Covered"].astype(bool)
            row[f"{size} priced"] = column.notna().mean()
            row[f"{size} covered"] = flag.mean()
        coverage[(regime, depth)] = row
pd.DataFrame(coverage).T.style.format("{:.1%}")

## 6. The two routes

Every statistic here has a second implementation, vectorised over the finished frame.  The
fold pays per message; `stats_from_frame` pays per column.

VWAP is the exception, and it is the point of the whole arrangement: a level shrinks by
cancellation as well as by execution, so no sequence of book configurations determines a
volume.  There is no from-frame route to compare against, and a session rebuilt from a
delta log has no `trades` at all.

In [7]:
routes = {}
for regime, s in REGIMES.items():
    book = TickArrayBook.for_prices(s.prices)
    session = MarketSession.from_occupied_levels(
        book, s.messages, DEPTH, SPEC, PRICE_UNIT, True
    )
    online = benchmark.time_fold(TickArrayBook, s, DEPTH, SPEC, True, REPEAT)
    plain = floor.loc["TickArrayBook", regime]
    vectorised = min(timeit.repeat(session.stats_from_frame, number=1, repeat=REPEAT))
    routes[regime] = {
        "fold, statistics off": plain,
        "fold, statistics on": online,
        "so the statistics cost": online - plain,
        "stats_from_frame": vectorised,
        "ratio": (online - plain) / vectorised,
    }
pd.DataFrame(routes).T.style.format({
    "fold, statistics off": "{:.3f} s", "fold, statistics on": "{:.3f} s",
    "so the statistics cost": "{:.3f} s", "stats_from_frame": "{:.3f} s",
    "ratio": "{:.1f}x",
})

,"fold, statistics off","fold, statistics on",so the statistics cost,stats_from_frame,ratio
shallow,0.249 s,1.143 s,0.894 s,0.046 s,19.4x
deep,0.257 s,1.240 s,0.983 s,0.046 s,21.4x


## 7. What the frame weighs, and what pandera costs

Two numbers that were already in this path before any statistic was added to it, and that
are larger than most of what was added.

`lobster_schema.validate` coerces to the nullable `Int64` extension dtype, which turns one
contiguous integer block into one masked column per field.  The frame gets wider in memory
and the call is not free.

In [8]:
s = REGIMES["shallow"]
session = MarketSession.from_occupied_levels(
    TickArrayBook.for_prices(s.prices), s.messages, DEPTH, SPEC, PRICE_UNIT, True
)
raw = session.lobster_book.astype("int64")
schema = MarketSession.lobster_schema(DEPTH)

pd.Series({
    "rows": len(session.lobster_book),
    "validate (s)": min(timeit.repeat(lambda: schema.validate(raw), number=1, repeat=REPEAT)),
    "fold with statistics (s)": benchmark.time_fold(
        TickArrayBook, s, DEPTH, SPEC, True, REPEAT
    ),
    "lobster_book as int64 (MB)": raw.memory_usage(deep=True).sum() / 1e6,
    "lobster_book as Int64 (MB)":
        session.lobster_book.memory_usage(deep=True).sum() / 1e6,
    "stats (MB)": session.stats.memory_usage(deep=True).sum() / 1e6,
    "trades (MB)": session.trades.memory_usage(deep=True).sum() / 1e6,
})

rows                          34802.000000
validate (s)                      0.014929
fold with statistics (s)          1.141101
lobster_book as int64 (MB)       11.415056
lobster_book as Int64 (MB)       12.807136
stats (MB)                       13.642384
trades (MB)                       3.619408
dtype: float64

## 8. The width of a statistics row

The buffer is sized to `row_columns()` and the frame to `columns()`: the rolling statistics
are made at assembly, so they are in the second and not the first.  Reading the widths off
the specification rather than counting them by hand is the only way to keep the estimate
honest as the specification changes.

In [9]:
widths = {}
for label, spec in SPECS.items():
    widths[label] = {
        "statistics buffer": len(spec.row_columns()),
        "statistics frame": len(spec.columns()),
        "trades buffer": len(spec.trade_row_columns()),
        "trades frame": len(spec.trade_columns()),
    }
table = pd.DataFrame(widths).T
table["frame, x old only"] = table["statistics frame"] / table.loc["old only", "statistics frame"]
table.style.format({"frame, x old only": "{:.2f}x"}, precision=0)

,statistics buffer,statistics frame,trades buffer,trades frame,"frame, x old only"
old only,27,27,3,3,1.00x
+ sweep,39,39,3,3,1.44x
+ windows,39,48,3,12,1.78x


## What to record

Whatever these cells say goes into `dev-context/market-microstructure.md`, **including
where it contradicts the reasoning that led to the design**.  The recording decision in
`AggregateBook.submit` was made on the strength of cell 2, and cell 4 is the argument for
reusing `SideStatistics.levels` rather than walking the side again; if either stops holding
on a later machine, the code should change rather than the note.